# SQL to Pandas Conversion Example

## CLAIMS_GOLD_001: Claims Aggregation by Policy

This notebook demonstrates how to convert the SQL query `CLAIMS_GOLD_001` to Python using pandas.

### Original SQL Query

```sql
SELECT 
    policy_id,
    SUM(claim_amount) AS total_claims,
    COUNT(claim_id) AS claim_count,
    AVG(claim_amount) AS avg_claim_amount,
    MAX(claim_amount) AS max_claim_amount,
    MIN(claim_amount) AS min_claim_amount
FROM claims_silver
GROUP BY policy_id
ORDER BY total_claims DESC;
```

### Conversion Steps

1. Load data into pandas DataFrame
2. Group by `policy_id`
3. Apply aggregations (SUM, COUNT, AVG, MAX, MIN)
4. Sort by `total_claims` descending


In [ ]:
# Step 1: Import libraries
import pandas as pd
import json
from pathlib import Path

print("✅ Libraries imported")


## Step 2: Load Sample Data

Load the claims data that would normally come from the `claims_silver` table.


In [ ]:
# Load sample claims data
data_path = Path("../data/raw_claims.json")

with open(data_path, "r") as f:
    data = json.load(f)

# Convert to DataFrame (simulating data from claims_silver table)
claims_df = pd.DataFrame(data["claims"])

# Convert claim_amount to float (it's a string in JSON)
claims_df["claim_amount"] = claims_df["claim_amount"].astype(float)

print(f"✅ Loaded {len(claims_df)} claims")
print(f"\nFirst few rows:")
claims_df.head()


## Step 3: Understand the SQL Logic

The SQL query:
- **Groups** by `policy_id`
- **Calculates** aggregations for each group:
  - `SUM(claim_amount)` → total_claims
  - `COUNT(claim_id)` → claim_count
  - `AVG(claim_amount)` → avg_claim_amount
  - `MAX(claim_amount)` → max_claim_amount
  - `MIN(claim_amount)` → min_claim_amount
- **Orders** by `total_claims DESC`

Let's see what policies we have:


In [ ]:
# Check unique policies
print(f"Unique policies: {claims_df['policy_id'].unique()}")
print(f"\nClaims per policy:")
claims_df.groupby('policy_id').size()


## Step 4: Convert SQL GROUP BY to Pandas

In SQL: `GROUP BY policy_id`

In Pandas: `df.groupby('policy_id')`


In [ ]:
# Group by policy_id (equivalent to SQL GROUP BY)
grouped = claims_df.groupby('policy_id')

print("✅ Grouped by policy_id")
print(f"Number of groups: {len(grouped)}")
print(f"\nGroups:")
for policy_id, group in grouped:
    print(f"  {policy_id}: {len(group)} claims")


## Step 5: Apply Aggregations

Convert SQL aggregations to pandas:

| SQL | Pandas |
|-----|--------|
| `SUM(claim_amount)` | `.agg({'claim_amount': 'sum'})` |
| `COUNT(claim_id)` | `.agg({'claim_id': 'count'})` |
| `AVG(claim_amount)` | `.agg({'claim_amount': 'mean'})` |
| `MAX(claim_amount)` | `.agg({'claim_amount': 'max'})` |
| `MIN(claim_amount)` | `.agg({'claim_amount': 'min'})` |


In [ ]:
# Method 1: Using .agg() with dictionary (most explicit)
result_df = claims_df.groupby('policy_id').agg({
    'claim_amount': ['sum', 'mean', 'max', 'min'],
    'claim_id': 'count'
})

# Flatten column names
result_df.columns = ['total_claims', 'avg_claim_amount', 'max_claim_amount', 
                     'min_claim_amount', 'claim_count']

# Reset index to make policy_id a column
result_df = result_df.reset_index()

print("✅ Aggregations complete")
result_df


## Step 6: Alternative - More Readable Approach

A cleaner way that matches the SQL more closely:


In [ ]:
# Method 2: More readable, step-by-step approach
result_df = (
    claims_df
    .groupby('policy_id')
    .agg(
        total_claims=('claim_amount', 'sum'),
        claim_count=('claim_id', 'count'),
        avg_claim_amount=('claim_amount', 'mean'),
        max_claim_amount=('claim_amount', 'max'),
        min_claim_amount=('claim_amount', 'min')
    )
    .reset_index()
)

print("✅ Aggregations complete (readable method)")
result_df


## Step 7: Apply ORDER BY

Convert SQL `ORDER BY total_claims DESC` to pandas `.sort_values()`


In [ ]:
# Sort by total_claims descending (equivalent to SQL ORDER BY total_claims DESC)
result_df = result_df.sort_values('total_claims', ascending=False)

print("✅ Sorted by total_claims DESC")
result_df


## Step 8: Complete Conversion - One Liner

Here's the complete SQL query converted to a single pandas chain:


In [ ]:
# Complete SQL to Pandas conversion in one chain
final_result = (
    claims_df
    .groupby('policy_id')
    .agg(
        total_claims=('claim_amount', 'sum'),
        claim_count=('claim_id', 'count'),
        avg_claim_amount=('claim_amount', 'mean'),
        max_claim_amount=('claim_amount', 'max'),
        min_claim_amount=('claim_amount', 'min')
    )
    .reset_index()
    .sort_values('total_claims', ascending=False)
)

print("✅ Complete conversion complete!")
print(f"\nFinal result ({len(final_result)} policies):")
final_result


## Step 9: Convert to Dictionary Format (for Pipeline)

The pipeline expects a list of dictionaries. Convert the DataFrame:


In [ ]:
# Convert DataFrame to list of dictionaries (what the pipeline expects)
result_list = final_result.to_dict('records')

print(f"✅ Converted to {len(result_list)} dictionaries")
print("\nFirst result:")
import json
print(json.dumps(result_list[0], indent=2))


## Step 10: Compare with Simple Aggregator

Let's verify our pandas result matches the simple aggregator:


In [ ]:
# Import the simple aggregator
import sys
sys.path.append('.')
from simple_aggregator import SimpleClaimsAggregator

# Convert DataFrame to list of dicts for aggregator
claims_list = claims_df.to_dict('records')

# Use simple aggregator
aggregator = SimpleClaimsAggregator()
aggregator_result = aggregator.aggregate_by_policy(claims_list)

# Compare results
print("Pandas result:")
print(f"  Policies: {len(result_list)}")
for r in result_list[:3]:
    print(f"    {r['policy_id']}: ${r['total_claims']:,.2f}")

print("\nSimple Aggregator result:")
print(f"  Policies: {len(aggregator_result)}")
for r in aggregator_result[:3]:
    print(f"    {r['policy_id']}: ${r['total_claims']:,.2f}")

# Verify they match
pandas_dict = {r['policy_id']: r['total_claims'] for r in result_list}
agg_dict = {r['policy_id']: r['total_claims'] for r in aggregator_result}

if pandas_dict == agg_dict:
    print("\n✅ Results match!")
else:
    print("\n❌ Results differ")


## Summary

### SQL to Pandas Conversion Cheat Sheet

| SQL | Pandas |
|-----|--------|
| `SELECT ... FROM table` | `df[['col1', 'col2']]` |
| `WHERE condition` | `df[df['col'] > value]` |
| `GROUP BY col` | `df.groupby('col')` |
| `SUM(col)` | `.agg({'col': 'sum'})` or `.sum()` |
| `COUNT(col)` | `.agg({'col': 'count'})` or `.count()` |
| `AVG(col)` | `.agg({'col': 'mean'})` or `.mean()` |
| `MAX(col)` | `.agg({'col': 'max'})` or `.max()` |
| `MIN(col)` | `.agg({'col': 'min'})` or `.min()` |
| `ORDER BY col DESC` | `.sort_values('col', ascending=False)` |
| `ORDER BY col ASC` | `.sort_values('col', ascending=True)` |

### Next Steps

1. **Use in Pipeline**: The `SimpleClaimsAggregator` class can be used directly in pipelines
2. **Add to Business Rules**: Move to `src/business_rules/aggregations.py` for production
3. **Test**: Write unit tests to ensure SQL and Python produce identical results
4. **Document**: Add to migration tracker: `sql_migration/sql_migration_tracker.md`
